In [ ]:
import os, glob
from natsort import natsorted

import cytoflow as flow
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from flow_analysis_tools import flow_violin
from utils import *
from utils.color_palettes import darken_palette
from utils import CONSTRUCT_NAME_DICT, GUIDE_NAME_DICT

from rclone_tools import NodeExperimentsDirectory

In [ ]:
DRIVE_RELPATH = "ARS_Working/Experiments/4_phHSC_Testing/260914_phHSCc12R1_Test/260928_phHSCc12R1_v2d10-BFP_v1d6-GFP_KD_check"
DATA_DIR = "/data_gilbert3/asathler/hsiung-box"
PROXY = "http://169.230.134.132:3128"

In [ ]:
Experiment = NodeExperimentsDirectory(
    drive="hsiung",
    source_relpath=DRIVE_RELPATH,
    node_mountpath=DATA_DIR,
    output_dirname="out",
    proxy=PROXY
)

In [ ]:
Experiment.stage()

In [ ]:
%matplotlib widget
matplotlib.rc('figure', dpi = 160)
plt.style.use('ggplot')

In [ ]:
FCS_FILES = sorted([Path(x) for x in glob.glob(os.path.join(Experiment.dest_dirpath, "*.fcs"))])
FCS_NAMES = [file.stem for file in FCS_FILES]
YLIMITS = (0, 7)
CHANNELS = {
    "VL1-A": ["Cas", "mTagBFP2"],
    "BL1-A": ["Guide", "eGFP"],
    "RL1-A": ["CD55", "APC"],
    "YL1-A": ["CD81", "PE"]
}
CHANNEL_CMAPS = {
    "VL1-A": black_blue_pastel_palette,
    "BL1-A": black_green_pastel_palette,
    "RL1-A": black_red_pastel_palette,
    "YL1-A": black_yellow_pastel_palette,
}
ALL_NEG_CONSTRUCT = "Experiment"
GATE_INFO = {}
print('\n'.join([f"'{x}'" for x in FCS_FILES]))
if isinstance(Experiment, NodeExperimentsDirectory):
    for x in FCS_FILES: Experiment.logger.info(x)

In [ ]:
CONSTRUCT_N_UNDERSCORE = -2
SECONDARY_N_UNDERSCORE = -1

def get_construct_name(name):
    spot = name.split("_")[-3]
    try:
        return CONSTRUCT_NAME_DICT[spot]
    except KeyError:
        return spot

def get_treatment_name(name):
    spot = name.split("_")[-2]
    try:
        return GUIDE_NAME_DICT[spot]
    except KeyError:
        return spot

def get_guide_name(name):
    spot = name.split("_")[-1]
    try:
        return GUIDE_NAME_DICT[spot]
    except KeyError:
        return spot

In [ ]:
max_fcs_len = max([len(x) for x in FCS_NAMES]) + 5
CONSTRUCT_NAMES = [get_construct_name(fcs_file) for fcs_file in FCS_NAMES]
TREATMENT_NAMES = [get_treatment_name(fcs_file) for fcs_file in FCS_NAMES]
GUIDE_NAMES = [get_guide_name(fcs_file) for fcs_file in FCS_NAMES]
for fcs, const, treatment, guide in zip(FCS_NAMES, CONSTRUCT_NAMES, TREATMENT_NAMES, GUIDE_NAMES):
    print_str = f"{fcs + ''.join([' '] * (max_fcs_len - len(fcs)))}{str(const):<20}{str(treatment):<20}{str(guide):<20}"
    print(print_str)
    # if isinstance(Experiment, NodeExperimentsDirectory):
    #     Experiment.logger.info(print_str)

import_op = flow.ImportOp(
    conditions = {
        "Construct": "category",
        "Treatment": "category",
        "Guide": "category"
    },
    tubes = [flow.Tube(file=fcs_file, conditions={
        "Construct": get_construct_name(fcs_name),
        "Treatment": get_treatment_name(fcs_name),
        "Guide": get_guide_name(fcs_name)
    }) for fcs_file, fcs_name in zip(FCS_FILES, FCS_NAMES)],
    # channels = channel_names
)

try:
    ex = import_op.apply()
except ValueError as e:
    error_text = str(e)
    if error_text == "Categorical categories cannot be null":
        raise Exception(
            "Functions 'get_construct_name' or 'get_guide_type' likely are turning None or NoneType." + \
            "\n\t   This is typically because a construct or guide name case isn't properly handled."
        )
    else:
        raise e

In [ ]:
import_op = flow.ImportOp(
    conditions = {
        "Construct": "category",
        "Guide": "category",
        "Treatment": "category"
    },
    tubes = [
        flow.Tube(file=fcs_file, conditions={
            "Construct": get_construct_name(fcs_file.stem),
            "Guide": get_treatment_name(str(fcs_file.stem)),
            "Treatment": get_guide_name(str(fcs_file.stem))
    }) for fcs_file in FCS_FILES],
    # channels = channel_names
)

ex = import_op.apply()

# Gating Single Cells

In [ ]:
single_gate = flow.PolygonOp(
    name="Single_Cell",
    xchannel="FSC-A",
    ychannel="SSC-A"
)

single_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True
).plot(ex, gridsize = 100)

In [ ]:
if isinstance(Experiment, NodeExperimentsDirectory):
    plt.savefig(Experiment.output_dirpath / f"single_cell_gate.png")

In [ ]:
GATE_INFO["single_cell_vertices"] = np.asarray(single_gate.vertices)
ex = single_gate.apply(ex)
plt.close()

# Single Gate Ops

In [ ]:
plt.ioff()

channels = list(CHANNELS)
subset = f"(Single_Cell == True) and (Construct == '{ALL_NEG_CONSTRUCT}')"
gates = {}

out = widgets.Output()
status = widgets.Label()
btn = widgets.Button(description="Accept threshold", button_style="success")

# Keep strong references: matplotlib holds event callbacks weakly, so if the
# view is garbage-collected, its click handler silently disappears.
state = {"i": 0, "op": None, "view": None, "fig": None}

def show_channel(i):
    chan = channels[i]
    op = flow.ThresholdOp(name=chan, channel=chan)
    view = op.default_view(huescale="log", interactive=True,
                           subset=subset, scale="log")
    plt.close("all")
    with out:
        out.clear_output(wait=True)
        view.plot(ex)
        fig = plt.gcf()
        display(fig.canvas)

    state.update(op=op, view=view, fig=fig)
    status.value = f"[{i+1}/{len(channels)}] {chan}: click the plot, then Accept"

def on_accept(_b):
    op = state["op"]
    if op.threshold is None:
        status.value = "No threshold set yet; click the plot first"
        return
    gates[op.channel] = op
    print(f"{op.channel}: {op.threshold}")
    state["i"] += 1
    if state["i"] < len(channels):
        show_channel(state["i"])
    else:
        plt.close(state["fig"])
        btn.disabled = True
        status.value = f"Done: {len(gates)} gates stored in `gates`"

btn.on_click(on_accept)
display(widgets.VBox([out, widgets.HBox([btn, status])]))
show_channel(0)
print(len(state["fig"].axes))

In [ ]:
for chan, thresh_op in gates.items():
    GATE_INFO[f"{chan}_threshold"] = thresh_op.threshold
    for char in ["-", "_"]:
        thresh_op.name = thresh_op.name.replace(char, "")
    ex = thresh_op.apply(ex)

In [ ]:
plt.ion()

# Quad Op

In [ ]:
quad_gate = flow.QuadOp(
    name="d_pos",
    xchannel="VL1-A",
    ychannel="BL1-A",
    # xscale="logicle",
    # yscale="logicle"
)

quad_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True,
    subset = f"(Single_Cell == True) & (Construct == '{ALL_NEG_CONSTRUCT}')",
    xscale="log", yscale="log"
).plot(ex, gridsize=100)

In [ ]:
plt.savefig(Experiment.output_dirpath / "quad_gate.png")
GATE_INFO["quad_gate"] = {
    "x": quad_gate.xthreshold,
    "y": quad_gate.ythreshold
}

In [ ]:
ex = quad_gate.apply(ex)
plt.close()

# Timeseries
Based on secondary

In [ ]:
%matplotlib inline

In [ ]:
plot_data = ex.data[ex.data["Single_Cell"] == True]
PLOT_CHANNEL = "VL1-A"
PALETTE_MAP = {
    "pARS001": black_blue_pastel_palette,
    "pCH45": black_green_pastel_palette,
    "pRA02": black_red_pastel_palette
}

In [ ]:
# fig, axes = plt.subplots(nrows=4, ncols=3)
output_channel = "YL1-A"
for chan in ["BL1-A", "VL1-A"]:
    facet_data = plot_data[
        ((plot_data["Guide"] == "T") | (plot_data["Guide"] == "NT")) &
        (plot_data["Construct"] != "Experiment")
    ].copy()
    for col in facet_data.select_dtypes(include=["category"]).columns:
        facet_data[col] = facet_data[col].cat.remove_unused_categories()

    g = sns.FacetGrid(
        data=facet_data,
        col="Treatment",
        row="Construct",
        hue="Guide",
        col_order=natsorted(facet_data["Treatment"].unique()),
        height=3,
        aspect=1,
        palette=sns.color_palette([darken_palette(black_red_pastel_palette)[0], CHANNEL_CMAPS[chan][1]]),
        legend_out=True
    )
    g.set(yscale="log", xscale="log")
    g.map(sns.scatterplot, chan, output_channel, alpha=0.2, edgecolor="w")
    g.set_titles("{row_name} | {col_name}µL", size=12)
    g.add_legend(title="Guide")

    if isinstance(Experiment, NodeExperimentsDirectory):
        plt.savefig(Experiment.output_dirpath / f"{output_channel}v{chan}_facet_grid_scatter.png")

In [ ]:
for chan in CHANNELS.keys():
    fig, axs = plt.subplot_mosaic(
        [
            ["Control", "Transduction1"],
            [".", "Transduction2"],
            [".", "Transduction3"]
        ],
        figsize=(12, 10),
        width_ratios=(2, 8),
        # sharex=True
        # sharey=True
        # layout="consrained"
    )

    flow_violin(
        data=plot_data.loc[(plot_data["Treatment"] == "B2") | (plot_data["Treatment"] == "B3")],
        x="Construct",
        # y=flow_channel,
        y=chan,
        hue="Guide",
        split=True,
        palette=CHANNEL_CMAPS[chan],
        log_y_axis=True,
        hue_order=sorted(plot_data.loc[(plot_data["Treatment"] == "B2") | (plot_data["Treatment"] == "B3")]["Guide"].unique()),
        ylabel="Controls",
        xlabel="",
        density_norm='width',
        ylim=YLIMITS,
        ax=axs['Control']
    )

    unique_constructs = plot_data.loc[(plot_data["Construct"] != "Experiment")]["Construct"].unique()
    for tdx, this_construct in enumerate(unique_constructs):
        ax_name = f'Transduction{tdx+1}'
        flow_violin(
            data=plot_data.loc[(plot_data["Construct"] == this_construct)],
            x="Treatment",
            y=chan,
            hue="Guide",
            log_y_axis=True,
            order=natsorted(plot_data.loc[(plot_data["Construct"] == this_construct)]["Treatment"].unique()),
            hue_order=plot_data.loc[(plot_data["Construct"] == this_construct)]["Guide"].unique(),
            split=True,
            palette=CHANNEL_CMAPS[chan],
            ylim=(0, 7),
            ax=axs[ax_name]
        )
        axs[ax_name].set_ylabel(this_construct)
        if tdx == len(unique_constructs) - 1: axs[ax_name].set_xlabel("Volume 1x viral supernatant (µL)")
        else: axs[ax_name].set_xlabel("")
    
    fig.suptitle(f"{CHANNELS[chan][1]} ({CHANNELS[chan][0]})")
    plt.savefig(Experiment.output_dirpath / f"{CHANNELS[chan][1]}-{CHANNELS[chan][0]}_all_cells.png")
    plt.tight_layout()



In [ ]:
GATE_COL = "BL1A"
GATE_VAL = True
assert GATE_COL in plot_data.select_dtypes(include=["category", "bool"]).columns, f"{GATE_COL} column not categorical or bool!"
d_pos = plot_data.loc[(plot_data[GATE_COL] == GATE_VAL)].copy()
for col in d_pos.select_dtypes(include=["category"]).columns:
        d_pos[col] = d_pos[col].cat.remove_unused_categories()
for chan in CHANNELS.keys():
    fig, axs = plt.subplot_mosaic(
        [
            ["Control", "Transduction1"],
            [".", "Transduction2"],
            [".", "Transduction3"]
        ],
        figsize=(12, 10),
        width_ratios=(2, 8),
        # sharex=True
        # sharey=True
        # layout="consrained"
    )

    flow_violin(
        data=plot_data.loc[(plot_data["Treatment"] == "B2") | (plot_data["Treatment"] == "B3")],
        x="Construct",
        # y=flow_channel,
        y=chan,
        hue="Guide",
        split=True,
        palette=CHANNEL_CMAPS[chan],
        log_y_axis=True,
        hue_order=sorted(plot_data.loc[(plot_data["Treatment"] == "B2") | (plot_data["Treatment"] == "B3")]["Guide"].unique()),
        ylabel="Control",
        xlabel="",
        density_norm='width',
        ylim=YLIMITS,
        ax=axs['Control']
    )

    unique_constructs = d_pos.loc[(d_pos["Construct"] != "Experiment")]["Construct"].unique()
    for tdx, this_construct in enumerate(unique_constructs):
        ax_name = f'Transduction{tdx+1}'
        flow_violin(
            data=d_pos.loc[(d_pos["Construct"] == this_construct)],
            x="Treatment",
            y=chan,
            hue="Guide",
            log_y_axis=True,
            order=natsorted(d_pos.loc[(d_pos["Construct"] == this_construct)]["Treatment"].unique()),
            hue_order=d_pos.loc[(d_pos["Construct"] == this_construct)]["Guide"].unique(),
            split=True,
            palette=CHANNEL_CMAPS[chan],
            density_norm="width",
            ylim=(0, 7),
            ax=axs[ax_name]
        )
        axs[ax_name].set_ylabel(this_construct)
        if tdx == len(unique_constructs) - 1: axs[ax_name].set_xlabel("Volume 1x viral supernatant (µL)")
        else: axs[ax_name].set_xlabel("")
    
    fig.suptitle(f"{CHANNELS[chan][1]} ({CHANNELS[chan][0]}) {GATE_COL} {GATE_VAL}")
    plt.savefig(Experiment.output_dirpath / f"{CHANNELS[chan][1]}-{CHANNELS[chan][0]}_{GATE_COL}_{GATE_VAL}.png")
    plt.tight_layout()



In [ ]:
plot_data.to_csv(Experiment.output_dirpath / f"processed_data.csv")
plot_data.to_csv(Experiment.output_dirpath / f"groupby_means.csv")

In [ ]:
np.savez(
    file=Experiment.output_dirpath / "gate_info.npz",
    **GATE_INFO
)

In [ ]:
Experiment.close()